In [2]:
import pandas as pd
print(pd.__version__)

3.0.5


In [3]:
import os 
print(os.getcwd())

e:\Coading\job-posting-authenticity-detector\notebooks


In [6]:
emscad = pd.read_csv("../data/emscad_core.csv")
synth = pd.read_csv("../data/synthetic_stress_test.csv")

In [7]:
print("EMSCAD columns:", emscad.columns.tolist())
print("\nSynthetic columns:", synth.columns.tolist())

EMSCAD columns: ['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']

Synthetic columns: ['job_id', 'job_title', 'job_description', 'requirements', 'benefits', 'company_name', 'company_profile', 'industry', 'employment_type', 'location', 'salary_range', 'required_experience_years', 'education_level', 'department', 'posting_date', 'application_deadline', 'contact_email', 'company_website', 'has_logo', 'num_open_positions', 'job_function', 'telecommuting', 'fraud_reason', 'text_length', 'is_fake']


In [19]:
rename_map = {
    "job_title": "title",
    "job_description" : "description",
    "has_logo" : "has_company_logo",
    "job_function": "function",
    "is_fake" : "fraudulent",
    "education_level" : "required_education"
}

In [9]:
synth_renamed = synth.rename(columns=rename_map)

In [10]:
emscad["source"] =  "emscad"
synth_renamed["source"] = "synthetic"

In [11]:
# This shows us exact categorical lables EMSCAD uses
print(emscad["required_experience"].value_counts(dropna=False))

required_experience
NaN                 7050
Mid-Senior level    3809
Entry level         2697
Associate           2297
Not Applicable      1116
Director             389
Internship           381
Executive            141
Name: count, dtype: int64


In [12]:
print(synth["required_experience_years"].describe())

count    3000.000000
mean        5.044000
std         3.168292
min         0.000000
25%         2.000000
50%         5.000000
75%         8.000000
max        10.000000
Name: required_experience_years, dtype: float64


## The Bucketing of data (years of experience-> job roles)

since the emscad required ewperience goes to
required_experience
NaN                 7050
Mid-Senior level    3809
Entry level         2697
Associate           2297
Not Applicable      1116
Director             389
Internship           381
Executive            141

but require experience years in synthetic data only goes to max 10 years , that means directorial roles are not achievable , so we bucket roles to experince insted of dividing the roles to fit into experience brackets . this will be more appropriate for the data analysis but some rows of synthetic wil remain unmapped.

Proposed boundaries
Years	EMSCAD label
0–1	Entry level
2–4	Associate
5–10	Mid-Senior level

In [22]:
#Bucket Function
def bucket_experience(years):
    if years <=1:
        return "Entry level"
    elif years <=4:
        return "Associate"
    else :
        return "Mid-Senior level"
    
synth_renamed["required_experience"] = synth_renamed["required_experience_years"].apply(bucket_experience)
# print(synth_renamed["required_experience"].value_counts())
print(synth_renamed.columns.tolist())

['job_id', 'title', 'description', 'requirements', 'benefits', 'company_name', 'company_profile', 'industry', 'employment_type', 'location', 'salary_range', 'required_experience_years', 'required_education', 'department', 'posting_date', 'application_deadline', 'contact_email', 'company_website', 'has_company_logo', 'num_open_positions', 'function', 'telecommuting', 'fraud_reason', 'text_length', 'fraudulent', 'source', 'required_experience']


In [24]:
#Empty set confirms it — every label in your bucketed synthetic column now exists exactly in EMSCAD's vocabulary
print(set(synth_renamed["required_experience"].unique())-set(emscad["required_experience"].unique()))

set()


In [15]:
emscad["source"] =  "emscad"
synth_renamed["source"] =  "synthetic"

In [25]:
print(emscad.info())
print(synth_renamed.info())

<class 'pandas.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   job_id               17880 non-null  int64
 1   title                17880 non-null  str  
 2   location             17534 non-null  str  
 3   department           6333 non-null   str  
 4   salary_range         2868 non-null   str  
 5   company_profile      14572 non-null  str  
 6   description          17879 non-null  str  
 7   requirements         15184 non-null  str  
 8   benefits             10668 non-null  str  
 9   telecommuting        17880 non-null  int64
 10  has_company_logo     17880 non-null  int64
 11  has_questions        17880 non-null  int64
 12  employment_type      14409 non-null  str  
 13  required_experience  10830 non-null  str  
 14  required_education   9775 non-null   str  
 15  industry             12977 non-null  str  
 16  function             11425 non-nu

In [17]:
print(synth_renamed.columns.tolist())

['job_id', 'title', 'description', 'requirements', 'benefits', 'company_name', 'company_profile', 'industry', 'employment_type', 'location', 'salary_range', 'required_experience_years', 'education_level', 'department', 'posting_date', 'application_deadline', 'contact_email', 'company_website', 'has_company_logo', 'num_open_positions', 'function', 'telecommuting', 'fraud_reason', 'text_length', 'fradulent', 'source', 'required_experience']


In [21]:
synth_renamed = synth.rename(columns=rename_map)
synth_renamed["source"] = "synthetic"
print(synth_renamed.columns.tolist())

['job_id', 'title', 'description', 'requirements', 'benefits', 'company_name', 'company_profile', 'industry', 'employment_type', 'location', 'salary_range', 'required_experience_years', 'required_education', 'department', 'posting_date', 'application_deadline', 'contact_email', 'company_website', 'has_company_logo', 'num_open_positions', 'function', 'telecommuting', 'fraud_reason', 'text_length', 'fraudulent', 'source']


# messiness of dataset
look at how differently missing values are distributed between the two datasets. EMSCAD has real gaps everywhere — salary_range only 2,868/17,880 filled, benefits 10,668/17,880, department 6,333/17,880. The synthetic dataset has almost no missing values except company_name, company_website, and fraud_reason. That's not a coincidence — it's a direct fingerprint of the two datasets' origins: EMSCAD is messy because real recruiters fill out forms inconsistently; the synthetic data is clean because whatever generated it didn't model that real-world sloppiness. That's actually a good observed insight to write down now — it strengthens your earlier point about treating synthetic data as a separate stress-test set rather than blending it in as if it were equally realistic.